# VCT 2026 Data Merge

Merges VCT 2026 match data files (Americas, EMEA, Pacific, China — Kickoff, Stage 1, Masters, etc.) into a single dataset, and adds a `MatchWinner` target column derived from map wins: whichever team (Team 1 or Team 2) won more maps is declared the winner.

**Requirements:** `pip install pandas openpyxl`

**Before running:** put all your VCT `.xlsx` files in one folder and update `INPUT_DIR` / `OUTPUT_PATH` in the config cell below.

In [15]:
!pip install openpyxl

In [16]:
import glob
import os
import pandas as pd

## Config — edit these two paths for your own machine

In [17]:
INPUT_DIR = r"C:\Users\Mahin\OneDrive\Desktop\SkbViProj"                          # folder containing your .xlsx files
OUTPUT_PATH = "VCT2026-Merged.xlsx"         # where the merged file will be saved

## 1. Locate and load every source file

In [18]:
def load_all_files(input_dir: str) -> list[pd.DataFrame]:
    files = sorted(glob.glob(os.path.join(input_dir, "*.xlsx")))
    if not files:
        raise FileNotFoundError(f"No .xlsx files found in '{input_dir}'")

    print(f"Found {len(files)} files:")
    for f in files:
        print(f"  - {f}")

    frames = []
    for f in files:
        df = pd.read_excel(f)
        df["SourceFile"] = os.path.basename(f)  # keep provenance, harmless extra col
        frames.append(df)

    # Sanity check: every file must share the same core schema before merging
    ref_cols = set(frames[0].columns) - {"SourceFile"}
    for f, df in zip(files, frames):
        cols = set(df.columns) - {"SourceFile"}
        if cols != ref_cols:
            missing = ref_cols - cols
            extra = cols - ref_cols
            raise ValueError(f"Schema mismatch in {f}: missing={missing}, extra={extra}")

    return frames

In [19]:
frames = load_all_files(INPUT_DIR)
merged = pd.concat(frames, ignore_index=True)
print(f"\nMerged shape (before target column): {merged.shape}")
merged.head()

Found 10 files:
  - C:\Users\Mahin\OneDrive\Desktop\SkbViProj\VCT2026-AmericasKickoff.xlsx
  - C:\Users\Mahin\OneDrive\Desktop\SkbViProj\VCT2026-AmericasStage1.xlsx
  - C:\Users\Mahin\OneDrive\Desktop\SkbViProj\VCT2026-ChinaKickoff.xlsx
  - C:\Users\Mahin\OneDrive\Desktop\SkbViProj\VCT2026-ChinaStage1.xlsx
  - C:\Users\Mahin\OneDrive\Desktop\SkbViProj\VCT2026-EMEAKickoff.xlsx
  - C:\Users\Mahin\OneDrive\Desktop\SkbViProj\VCT2026-MastersSantiago.xlsx
  - C:\Users\Mahin\OneDrive\Desktop\SkbViProj\VCT2026-PacificKickoff.xlsx
  - C:\Users\Mahin\OneDrive\Desktop\SkbViProj\VCT2026-PacificStage1.xlsx
  - C:\Users\Mahin\OneDrive\Desktop\SkbViProj\VCT26-EMEAStage1.xlsx
  - C:\Users\Mahin\OneDrive\Desktop\SkbViProj\VCT26-MastersLondon.xlsx

Merged shape (before target column): (342, 53)


,ID,Region,Tournament,Patch,Team 1,Team 2,Phase,Map Numbers,Team1Ban1,Team2Ban1,...,M4T1,M4T2,Map 5,Map 5 Winner,M5R1,M5R2,M5Half,M5T1,M5T2,SourceFile
0,160126400,Americas,Kickoff,12.0,ENVY,EVIL GENIUSES,Main Event,3,Haven,Breeze,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,VCT2026-AmericasKickoff.xlsx
1,160126710,Americas,Kickoff,12.0,LOUD,CLOUD9,Main Event,3,Split,Breeze,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,VCT2026-AmericasKickoff.xlsx
2,170126400,Americas,Kickoff,12.0,KRU ESPORTS,FURIA,Main Event,3,Haven,Split,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,VCT2026-AmericasKickoff.xlsx
3,170126705,Americas,Kickoff,12.0,100 THIEVES,LEVIATAN,Main Event,3,Corrode,Abyss,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,VCT2026-AmericasKickoff.xlsx
4,180126400,Americas,Kickoff,12.0,NRG,ENVY,Main Event,3,Haven,Corrode,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,VCT2026-AmericasKickoff.xlsx


## 2. Compute `MatchWinner` from per-map winners

In [20]:
MAP_WINNER_COLS = [f"Map {i} Winner" for i in range(1, 6)]


def compute_match_winner(row: pd.Series):
    team1, team2 = row["Team 1"], row["Team 2"]
    t1_maps, t2_maps = 0, 0
    for col in MAP_WINNER_COLS:
        winner = row[col]
        if pd.isna(winner):
            continue
        if winner == team1:
            t1_maps += 1
        elif winner == team2:
            t2_maps += 1

    if t1_maps == 0 and t2_maps == 0:
        return None  # no map data available for this match at all
    if t1_maps > t2_maps:
        return team1
    if t2_maps > t1_maps:
        return team2
    return None  # tied map count -- shouldn't happen in a completed Bo3/Bo5


def add_match_winner(df: pd.DataFrame) -> pd.DataFrame:
    df["Team1MapsWon"] = df.apply(
        lambda r: sum(r[c] == r["Team 1"] for c in MAP_WINNER_COLS if pd.notna(r[c])),
        axis=1,
    )
    df["Team2MapsWon"] = df.apply(
        lambda r: sum(r[c] == r["Team 2"] for c in MAP_WINNER_COLS if pd.notna(r[c])),
        axis=1,
    )
    df["MatchWinner"] = df.apply(compute_match_winner, axis=1)
    return df

In [21]:
merged = add_match_winner(merged)
merged[["ID", "Region", "Tournament", "Team 1", "Team 2", "Team1MapsWon", "Team2MapsWon", "MatchWinner"]].head(10)

,ID,Region,Tournament,Team 1,Team 2,Team1MapsWon,Team2MapsWon,MatchWinner
0,160126400,Americas,Kickoff,ENVY,EVIL GENIUSES,2,1,ENVY
1,160126710,Americas,Kickoff,LOUD,CLOUD9,0,2,CLOUD9
2,170126400,Americas,Kickoff,KRU ESPORTS,FURIA,1,2,FURIA
3,170126705,Americas,Kickoff,100 THIEVES,LEVIATAN,2,1,100 THIEVES
4,180126400,Americas,Kickoff,NRG,ENVY,2,0,NRG
5,180126600,Americas,Kickoff,MIBR,ENVY,2,0,MIBR
6,190126400,Americas,Kickoff,SENTINELS,FURIA,0,2,FURIA
7,190126620,Americas,Kickoff,G2 ESPORTS,100 THIEVES,2,1,G2 ESPORTS
8,240126400,Americas,Kickoff,LOUD,100 THIEVES,1,2,100 THIEVES
9,240126710,Americas,Kickoff,EVIL GENIUSES,SENTINELS,0,2,SENTINELS


## 3. Check for unresolved matches (missing/incomplete map data)

In [22]:
unresolved = merged[merged["MatchWinner"].isna()]
if len(unresolved):
    print(f"WARNING: {len(unresolved)} matches have no resolvable MatchWinner:")
    display(unresolved[["ID", "Region", "Tournament", "Team 1", "Team 2"]])
else:
    print("All matches resolved successfully.")

All matches resolved successfully.


## 4. Save merged dataset

In [23]:
merged.to_excel(OUTPUT_PATH, index=False)
print(f"Saved merged dataset to: {OUTPUT_PATH}")
print(f"Final shape: {merged.shape}")

Saved merged dataset to: VCT2026-Merged.xlsx
Final shape: (342, 56)


## 5. Quick sanity check summaries

In [24]:
print("Matches per Region:")
print(merged["Region"].value_counts())
print()
print("Matches per Tournament:")
print(merged["Tournament"].value_counts())

Matches per Region:
Region
China       78
Americas    72
EMEA        72
Pacific     72
Global      48
Name: count, dtype: int64

Matches per Tournament:
Tournament
Stage1              174
Kickoff             120
Masters Santiago     24
Masters London       24
Name: count, dtype: int64
